# Assignment 6 — Linear Regression for Final Grade (G3) Prediction
**Dataset:** UCI Student Performance (`student-por.csv`)  
**Target:** continuous `G3` (0–20); inputs exclude G1/G2/G3 (early prediction)

**Aim:** Build a preprocessing pipeline + Linear Regression, report coefficients, MAE/MSE/RMSE/R², and plot actual-vs-predicted, residuals and residual distribution.


## 1. Library imports

In [1]:
import io, os
import urllib.request, zipfile
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("outputs", exist_ok=True)
print("random_state =", RANDOM_STATE)

random_state = 42


## 2. Dataset loading (official UCI source)

In [2]:
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
raw = urllib.request.urlopen(UCI_URL, timeout=60).read()
zf = zipfile.ZipFile(io.BytesIO(raw))
with zf.open("student-por.csv") as f:
    df = pd.read_csv(f, sep=";")
print("Shape:", df.shape)

Shape: (649, 33)


## 3. Dataset preview

In [3]:
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [4]:
print(df["G3"].describe())
print("\nMissing total:", int(df.isnull().sum().sum()))

count    649.000000
mean      11.906009
std        3.230656
min        0.000000
25%       10.000000
50%       12.000000
75%       14.000000
max       19.000000
Name: G3, dtype: float64

Missing total: 0


## 4. Data preprocessing — pipeline (exclude G1/G2 from inputs, G3 = target)

In [5]:
X_raw = df.drop(columns=["G1", "G2", "G3"])
y = df["G3"].to_numpy()
cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_raw.select_dtypes(exclude=["object"]).columns.tolist()
print(f"Inputs: {X_raw.shape[1]} (numeric={len(num_cols)}, categorical={len(cat_cols)})")
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train {X_train_raw.shape} Test {X_test_raw.shape}")

Inputs: 30 (numeric=13, categorical=17)
Train (519, 30) Test (130, 30)


C:\Users\ns479\AppData\Local\Temp\ipykernel_532\2589407183.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()


## 5. Model implementation — Linear Regression

In [6]:
pipe = Pipeline([("prep", preprocess), ("lr", LinearRegression())])
pipe.fit(X_train_raw, y_train)
print("Intercept:", round(float(pipe.named_steps['lr'].intercept_), 4))
feat_names = list(pipe.named_steps["prep"].get_feature_names_out())
coefs = pipe.named_steps["lr"].coef_
coef_df = pd.DataFrame({"feature": feat_names, "coefficient": coefs}).sort_values(
    "coefficient", key=np.abs, ascending=False)
print(coef_df.head(15).round(4).to_string(index=False))
coef_df.to_csv("outputs/coefficients.csv", index=False)

Intercept: 10.3842
               feature  coefficient
         num__failures      -0.8841
       cat__higher_yes       0.8435
        cat__higher_no      -0.8435
    cat__schoolsup_yes      -0.8017
     cat__schoolsup_no       0.8017
        cat__school_GP       0.6968
        cat__school_MS      -0.6968
     cat__Fjob_at_home       0.6419
      cat__Mjob_health       0.6017
     cat__reason_other      -0.5656
    cat__Fjob_services      -0.4574
   cat__guardian_other       0.4561
     cat__Mjob_at_home      -0.4432
cat__reason_reputation       0.4368
      cat__Fjob_health      -0.3992


## 6. Evaluation — MAE, MSE, RMSE, R²

In [7]:
for split, Xr, yr in [("Train", X_train_raw, y_train), ("Test", X_test_raw, y_test)]:
    p = pipe.predict(Xr)
    mae = mean_absolute_error(yr, p); mse = mean_squared_error(yr, p)
    print(f"{split}: MAE={mae:.4f} MSE={mse:.4f} RMSE={np.sqrt(mse):.4f} R2={r2_score(yr, p):.4f}")
y_pred = pipe.predict(X_test_raw)
mae = mean_absolute_error(y_test, y_pred); mse = mean_squared_error(y_test, y_pred)
metrics = pd.DataFrame([{"MAE": mae, "MSE": mse, "RMSE": float(np.sqrt(mse)), "R2": r2_score(y_test, y_pred)}])
print(metrics.round(4))
metrics.round(4).to_csv("outputs/metrics.csv", index=False)

Train: MAE=1.8320 MSE=6.4476 RMSE=2.5392 R2=0.3874
Test: MAE=2.1564 MSE=8.1898 RMSE=2.8618 R2=0.1602
      MAE     MSE    RMSE      R2
0  2.1564  8.1898  2.8618  0.1602


## 7. Tables and graphs — actual vs predicted, residuals, residual distribution

In [8]:
resid = y_test - y_pred
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
axes[0].scatter(y_test, y_pred, alpha=0.55, s=20)
lo, hi = float(min(y_test.min(), y_pred.min())), float(max(y_test.max(), y_pred.max()))
axes[0].plot([lo, hi], [lo, hi], "r--", label="Ideal")
axes[0].set_xlabel("Actual G3"); axes[0].set_ylabel("Predicted G3")
axes[0].set_title(f"Actual vs predicted (R2={r2_score(y_test, y_pred):.3f})"); axes[0].legend()
axes[1].scatter(y_pred, resid, alpha=0.55, s=20)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted G3"); axes[1].set_ylabel("Residual (actual-pred)")
axes[1].set_title("Residual plot")
axes[2].hist(resid, bins=20, edgecolor="black")
axes[2].set_xlabel("Residual"); axes[2].set_ylabel("Count"); axes[2].set_title("Residual distribution")
plt.suptitle("Linear Regression diagnostics")
plt.tight_layout(); plt.savefig("outputs/diagnostics.png", dpi=150); plt.close()
print("Saved: outputs/diagnostics.png; mean resid:", round(float(resid.mean()), 4))

Saved: outputs/diagnostics.png; mean resid: 0.181


## 8. Result and conclusion
- `ColumnTransformer(StandardScaler + OneHotEncoder)` + `LinearRegression`; 80/20 split `random_state=42`.
- Coefficients (`outputs/coefficients.csv`), metrics (`outputs/metrics.csv`) and diagnostics (`outputs/diagnostics.png`) all computed at runtime.
